# Single Pixel Camera + Gaussian Splatting
## Demo realista con una vista del dataset `truck`

Esta demo mezcla los dos notebooks anteriores: usa una imagen real del dataset `truck`, simula mediciones de una Single Pixel Camera con patrones Hadamard y reconstruye la escena mediante una suma diferenciable de gaussianas 2D.

La etapa es deliberadamente 2D: sirve para demostrar el vínculo entre las mediciones SPC y las gaussianas sin depender todavía de COLMAP ni de extensiones CUDA. La información de la pose COLMAP se conserva y se verifica para preparar la extensión a varias vistas y gaussianas 3D.

In [ ]:
from pathlib import Path
import re
import struct
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from scipy.linalg import hadamard
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass

# Configuración de la demo. Se puede cambiar IMAGE_PATH manualmente.
IMAGE_PATH = None
SIZE = 64
CHANNEL_MODE = 'rgb'  # 'rgb' o 'gray'
RATIOS = [0.05, 0.25, 0.50]
N_GAUSSIANS = 256
ITERATIONS = 1000
PRIOR_WEIGHT = 0.35
SEED = 449
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('Dispositivo:', DEVICE)
print('Resolución de trabajo:', SIZE, 'x', SIZE, '| canales:', CHANNEL_MODE)

In [ ]:
def find_truck_image():
    roots = [
        Path('/content/truck'),
        Path('/content/drive/MyDrive/inf479/tandt_db_extracted'),
        Path('/content/drive/MyDrive/inf479'),
    ]
    preferred_names = {'000001.jpg', '01.jpg', '000001.png', '01.png'}
    for root in roots:
        if not root.exists():
            continue
        candidates = sorted(
            (path for path in root.rglob('*')
             if path.is_file() and path.parent.name.lower() == 'images'
             and path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}),
            key=lambda path: (path.name not in preferred_names, str(path))
        )
        if candidates:
            return candidates[0]
    raise FileNotFoundError('No encontré imágenes de truck. Ejecuta primero la preparación del dataset o define IMAGE_PATH.')

if IMAGE_PATH is None:
    IMAGE_PATH = find_truck_image()
else:
    IMAGE_PATH = Path(IMAGE_PATH)
IMAGE_PATH = Path(IMAGE_PATH)
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f'No existe IMAGE_PATH: {IMAGE_PATH}')

with Image.open(IMAGE_PATH) as source:
    original_rgb = np.asarray(source.convert('RGB'), dtype=np.float32) / 255.0
image_array = original_rgb.mean(axis=-1, keepdims=True) if CHANNEL_MODE == 'gray' else original_rgb
image_uint8 = np.round(image_array * 255).astype(np.uint8)
if image_uint8.shape[-1] == 1:
    image_uint8 = image_uint8[..., 0]
resized = np.asarray(Image.fromarray(image_uint8).resize((SIZE, SIZE), Image.Resampling.LANCZOS), dtype=np.float32) / 255.0
if resized.ndim == 2:
    resized = resized[..., None]
img = resized

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original_rgb); axes[0].set_title(f'Original truck\n{IMAGE_PATH.name}'); axes[0].axis('off')
axes[1].imshow(img[..., 0] if img.shape[-1] == 1 else img); axes[1].set_title(f'Entrada SPC\n{SIZE}x{SIZE}'); axes[1].axis('off')
plt.tight_layout(); plt.show()
print('Imagen usada:', IMAGE_PATH)

## Pose COLMAP asociada

La demo 2D no necesita la pose para ejecutarse, pero verificamos que la imagen pertenece a una escena COLMAP. Esto deja trazabilidad para la futura extensión a varias vistas.

In [ ]:
def read_colmap_image_names(images_bin):
    names = []
    with open(images_bin, 'rb') as file:
        count = struct.unpack('<Q', file.read(8))[0]
        for _ in range(count):
            header = file.read(68)
            if len(header) != 68:
                raise RuntimeError(f'images.bin truncado: {images_bin}')
            name_bytes = bytearray()
            while True:
                byte = file.read(1)
                if byte == b'\x00':
                    break
                if not byte:
                    raise RuntimeError(f'Nombre COLMAP incompleto: {images_bin}')
                name_bytes.extend(byte)
            n_points = struct.unpack('<Q', file.read(8))[0]
            file.seek(24 * n_points, 1)
            names.append(name_bytes.decode('utf-8'))
    return names

def numeric_key(name):
    stem = Path(name).stem.lower()
    match = re.search(r'(\d+)$', stem)
    return ('numeric', str(int(match.group(1)))) if match else ('text', stem)

scene_root = next((parent for parent in [IMAGE_PATH.parent, *IMAGE_PATH.parents] if (parent / 'sparse' / '0' / 'images.bin').exists()), None)
pose_match = None
if scene_root is not None:
    pose_names = read_colmap_image_names(scene_root / 'sparse' / '0' / 'images.bin')
    pose_match = next((name for name in pose_names if numeric_key(name) == numeric_key(IMAGE_PATH.name)), None)
print('Escena COLMAP:', scene_root or 'no encontrada')
print('Pose asociada:', pose_match or 'no encontrada; la demo 2D puede continuar')

## Mediciones SPC con Hadamard

Cada medición es un producto interno entre una máscara Hadamard y la imagen. La reconstrucción clásica deja en cero los coeficientes no medidos.

In [ ]:
h = (hadamard(SIZE) / np.sqrt(SIZE)).astype(np.float32)
changes = np.sum(h[:, 1:] != h[:, :-1], axis=1)
order_1d = np.argsort(changes, kind='stable')
pairs = [(i, j) for i in order_1d for j in order_1d]
pairs.sort(key=lambda ij: (changes[ij[0]] + changes[ij[1]], max(changes[ij[0]], changes[ij[1]])))
Ht = torch.tensor(h, device=DEVICE)
x_true = torch.tensor(img, device=DEVICE)

def hadamard_coefficients(x):
    if x.ndim == 2:
        x = x[..., None]
    return torch.einsum('ai,ijc,bj->abc', Ht, x, Ht)

def measure(x, n):
    coeffs = hadamard_coefficients(x)
    idx = torch.tensor(pairs[:n], device=DEVICE, dtype=torch.long)
    return coeffs[idx[:, 0], idx[:, 1]], idx

def reconstruct_hadamard(y, idx):
    coeffs = torch.zeros((SIZE, SIZE, x_true.shape[-1]), device=DEVICE)
    coeffs[idx[:, 0], idx[:, 1]] = y
    return torch.einsum('ai,ijc,bj->abc', Ht.T, coeffs, Ht).clamp(0, 1)

print('Mediciones completas por canal:', SIZE * SIZE)

## Gaussian Splatting 2D diferenciable

La imagen se representa como una suma de gaussianas con posición, escala, rotación e intensidad RGB. Los parámetros se optimizan para reproducir las mismas mediciones SPC y se estabilizan con una regularización suave hacia la reconstrucción Hadamard. Así evitamos que una solución aleatoria de un problema subdeterminado produzca manchas deformadas.

In [ ]:
gy, gx = torch.meshgrid(torch.arange(SIZE, device=DEVICE).float(), torch.arange(SIZE, device=DEVICE).float(), indexing='ij')
grid = torch.stack([gx, gy], dim=-1)

def inverse_softplus(value):
    return np.log(np.expm1(value))

def fit_gaussians(y, idx, target_prior, channels, k=N_GAUSSIANS, iters=ITERATIONS, seed=SEED):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    side = int(np.ceil(np.sqrt(k)))
    coordinates = torch.linspace(1, SIZE - 2, side, device=DEVICE)
    mesh_y, mesh_x = torch.meshgrid(coordinates, coordinates, indexing='ij')
    centers = torch.stack([mesh_x.reshape(-1), mesh_y.reshape(-1)], dim=-1)[:k]
    normalized_centers = (centers / (SIZE - 1)).clamp(1e-4, 1 - 1e-4)
    raw_mu = torch.logit(normalized_centers).requires_grad_()
    log_s = torch.full((k, 2), float(np.log(2.5)), device=DEVICE).requires_grad_()
    theta = torch.zeros(k, device=DEVICE).requires_grad_()
    initial_amp = (target_prior.mean(dim=(0, 1)) / 5).clamp_min(1e-3)
    raw_amp = torch.log(torch.expm1(initial_amp)).repeat(k, 1).requires_grad_()

    def render():
        mu = torch.sigmoid(raw_mu) * (SIZE - 1)
        scales = torch.exp(log_s.clamp(np.log(0.5), np.log(SIZE / 2)))
        amp = torch.nn.functional.softplus(raw_amp)
        c, s = torch.cos(theta), torch.sin(theta)
        rot = torch.stack([torch.stack([c, -s], -1), torch.stack([s, c], -1)], -2)
        d = grid[None] - mu[:, None, None, :]
        d = torch.einsum('kij,khwj->khwi', rot.transpose(1, 2), d)
        q = (d / scales[:, None, None, :]) ** 2
        basis = torch.exp(-0.5 * q.sum(-1))
        return (basis[..., None] * amp[:, None, None, :]).sum(0)

    opt = torch.optim.Adam([
        {'params': raw_mu, 'lr': 0.08},
        {'params': log_s, 'lr': 0.02},
        {'params': theta, 'lr': 0.02},
        {'params': raw_amp, 'lr': 0.05},
    ])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, iters)
    measurement_denominator = y.square().mean().clamp_min(1e-8)
    prior_denominator = target_prior.square().mean().clamp_min(1e-8)
    history = []
    for step in range(iters):
        opt.zero_grad()
        rendered = render()
        coeffs = hadamard_coefficients(rendered)
        prediction = coeffs[idx[:, 0], idx[:, 1]]
        measurement_loss = (prediction - y).square().mean() / measurement_denominator
        prior_loss = (rendered.clamp(0, 1) - target_prior).square().mean() / prior_denominator
        loss = measurement_loss + PRIOR_WEIGHT * prior_loss
        loss.backward()
        opt.step(); sched.step()
        if step == 0 or (step + 1) % max(1, iters // 10) == 0:
            history.append(float(loss.detach().cpu()))
    return render().detach().clamp(0, 1), history

## Barrido de ratios y comparación

In [ ]:
def metric_pair(reference, reconstruction):
    ref = reference.detach().cpu().numpy()
    rec = reconstruction.detach().cpu().numpy()
    if ref.shape[-1] == 1:
        ref, rec = ref[..., 0], rec[..., 0]
        return peak_signal_noise_ratio(ref, rec, data_range=1.0), structural_similarity(ref, rec, data_range=1.0)
    return peak_signal_noise_ratio(ref, rec, data_range=1.0), structural_similarity(ref, rec, channel_axis=-1, data_range=1.0)

rows, reconstructions, histories = [], {}, {}
for ratio in RATIOS:
    n = max(1, int(round(ratio * SIZE * SIZE)))
    y, idx = measure(x_true, n)
    hadamard_rec = reconstruct_hadamard(y, idx)
    start = time.time()
    gaussian_rec, history = fit_gaussians(y, idx, hadamard_rec.detach(), x_true.shape[-1], seed=SEED + int(ratio * 1000))
    elapsed = time.time() - start
    reconstructions[ratio] = (hadamard_rec, gaussian_rec)
    histories[ratio] = history
    for method, reconstruction in [('Hadamard clásico', hadamard_rec), ('Gaussian Splatting 2D', gaussian_rec)]:
        psnr, ssim = metric_pair(x_true, reconstruction)
        rows.append({'ratio': ratio, 'mediciones': n, 'método': method, 'PSNR (dB)': round(psnr, 2), 'SSIM': round(ssim, 3)})
    print(f'ratio={ratio:.0%} | mediciones={n} | tiempo gaussianas={elapsed:.1f}s | pérdida final={history[-1]:.5f}')

results = pd.DataFrame(rows)
display(results.pivot(index=['ratio', 'mediciones'], columns='método', values=['PSNR (dB)', 'SSIM']))
results.to_csv('/content/spc_gaussian_truck_results.csv', index=False)
print('Resultados guardados en /content/spc_gaussian_truck_results.csv')

In [ ]:
def show_image(tensor):
    array = tensor.detach().cpu().numpy()
    return array[..., 0] if array.shape[-1] == 1 else array

fig, axes = plt.subplots(2, len(RATIOS) + 1, figsize=(4 * (len(RATIOS) + 1), 8))
axes[0, 0].imshow(show_image(x_true), interpolation='bilinear'); axes[0, 0].set_title('Original truck')
axes[1, 0].imshow(show_image(x_true), interpolation='bilinear'); axes[1, 0].set_title('Referencia')
for col, ratio in enumerate(RATIOS, start=1):
    hadamard_rec, gaussian_rec = reconstructions[ratio]
    axes[0, col].imshow(show_image(hadamard_rec), interpolation='bilinear'); axes[0, col].set_title(f'Hadamard {ratio:.0%}')
    axes[1, col].imshow(show_image(gaussian_rec), interpolation='bilinear'); axes[1, col].set_title(f'Gaussianas 2D {ratio:.0%}')
for ax in axes.ravel(): ax.axis('off')
fig.suptitle('Single Pixel Camera + Gaussian Splatting 2D | dataset truck', y=1.02)
plt.tight_layout()
fig.savefig('/content/spc_gaussian_truck_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada en /content/spc_gaussian_truck_demo.png')

## Interpretación y extensión a 3D

- El notebook demuestra el vínculo entre la adquisición SPC y una representación gaussiana: la pérdida se calcula en el dominio de mediciones.
- La comparación debe hacerse entre los mismos ratios y contra la imagen original conocida.
- Esta etapa todavía es 2D: no inventa profundidad ni nuevas vistas.
- Para extenderla a 3D se necesitan varias imágenes del mismo `truck`, sus poses COLMAP y un único conjunto compartido de gaussianas 3D. La pérdida por vista sería $\|\Phi_v(R(G, c_v)) - y_v\|^2$, donde $c_v$ es la cámara de la vista $v$.

La demo guarda además la imagen y la pose COLMAP asociada, de modo que la siguiente etapa puede reutilizar exactamente estas vistas.